In [4]:
import os

print(os.getcwd())

/home/ubuntu/spark-notebooks/NYC311-Pluto/notebooks


In [5]:
import os
from datetime import date

import findspark
findspark.init()

from pyspark.sql import functions as F

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col, lit, trim, length, count, countDistinct, coalesce, desc, when,
    try_to_timestamp, year,
    sum as spark_sum,
    min as spark_min,
    max as spark_max,
    round as spark_round
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType
)

spark = (
    SparkSession.builder \
    .appName("DAT535-Project-NYC311-silver2") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "5g") \
    .getOrCreate()
)
print("Spark is ready!")
print("Spark version:", spark.version)
spark.sparkContext.setLogLevel("WARN")


In [23]:
spark = (
    SparkSession.builder
    .appName("DAT535-Project-NYC311-silver2")
    .master("spark://192.168.18.168:7077")
    .config("spark.driver.host", "192.168.18.168")
    .config("spark.cores.max", "8")
    .config("spark.executor.cores", "4")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Connected Master:", spark.sparkContext.master)

Spark version: 3.5.0
Connected Master: spark://192.168.18.168:7077


26/10/30 12:31:19 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


load dataframe

In [11]:
# Path to the original NYC 311 dataset
nyc311_path = "/home/ubuntu/spark-project-data/bronze/nyc311/nyc311_2020_present.csv"

# Reload the raw dataset
nyc311_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(nyc311_path)
)

print("NYC 311 raw DataFrame restored successfully!")
print("Number of columns:", len(nyc311_raw_df.columns))

NYC 311 raw DataFrame restored successfully!
Number of columns: 44


# Read the NYC-311 raw CSV

In [9]:
nyc311_raw_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv("/home/ubuntu/spark-project-data/bronze/nyc311/nyc311_2020_present.csv")

print("NYC 311 dataset loaded!")
print("Number of columns:", len(nyc311_raw_df.columns))
print(nyc311_raw_df.columns)

NYC 311 dataset loaded!
Number of columns: 44
['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Additional Details', 'Location Type', 'Incident Zip', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'City', 'Landmark', 'Facility Type', 'Status', 'Due Date', 'Resolution Description', 'Resolution Action Updated Date', 'Community Board', 'Council District', 'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Open Data Channel Type', 'Park Facility Name', 'Park Borough', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment', 'Latitude', 'Longitude', 'Location']


# 3. Inspect Unique Key and BBL

In [21]:
nyc311_raw_df.select(
    "Unique Key",
    "BBL"
).show(20, truncate=False)

# complaint about noise on a street may not have a specific property BBL.

[Stage 34:>                                                         (0 + 1) / 1]

+----------+----------+
|Unique Key|BBL       |
+----------+----------+
|70702319  |NULL      |
|70709493  |4037310065|
|70703577  |5070830014|
|70698764  |4092350019|
|70701844  |4092840019|
|70709883  |3083000041|
|70698017  |3066740023|
|70699373  |3012820063|
|70703497  |4054390060|
|70705934  |3001460014|
|70700470  |5033380056|
|70708978  |1004630015|
|70702099  |1018380037|
|70704393  |3012400038|
|70703417  |4034630001|
|70704490  |1005390020|
|70699447  |NULL      |
|70709672  |1015510048|
|70707498  |3002510011|
|70704800  |NULL      |
+----------+----------+
only showing top 20 rows



# 4. Unique Key → request_id

In [22]:
nyc311_fields_df = nyc311_raw_df.withColumn(
    "request_id",
    F.trim(F.col("Unique Key"))
)

nyc311_fields_df.select(
    "Unique Key",
    "request_id"
).show(10, truncate=False)

+----------+----------+
|Unique Key|request_id|
+----------+----------+
|70702319  |70702319  |
|70709493  |70709493  |
|70703577  |70703577  |
|70698764  |70698764  |
|70701844  |70701844  |
|70709883  |70709883  |
|70698017  |70698017  |
|70699373  |70699373  |
|70703497  |70703497  |
|70705934  |70705934  |
+----------+----------+
only showing top 10 rows



# 5. Validate request ID formats

In [23]:
# A valid request ID should contain only digits.

nyc311_fields_df.groupBy(
    F.when(
        F.col("request_id").isNull() |
        (F.col("request_id") == ""),
        "Missing"
    ).when(
        F.col("request_id").rlike(r"^[0-9]+$"),
        "Valid"
    ).otherwise("Invalid").alias("request_id_status")
).count().show()

[Stage 36:=====================================================>(118 + 2) / 120]

+-----------------+--------+
|request_id_status|   count|
+-----------------+--------+
|            Valid|22761196|
+-----------------+--------+



# 6. Create a cleaned BBL column

In [24]:
nyc311_fields_df = (
    nyc311_fields_df
    .withColumn("bbl_raw", F.col("BBL"))
    .withColumn(
        "bbl",
        F.when(
            F.trim(F.col("BBL")).rlike(r"^[1-5][0-9]{9}$"),
            F.trim(F.col("BBL"))
        ).otherwise(F.lit(None).cast("string"))
    )
)

nyc311_fields_df.select(
    "BBL",
    "bbl_raw",
    "bbl"
).show(20, truncate=False)

+----------+----------+----------+
|BBL       |bbl_raw   |bbl       |
+----------+----------+----------+
|NULL      |NULL      |NULL      |
|4037310065|4037310065|4037310065|
|5070830014|5070830014|5070830014|
|4092350019|4092350019|4092350019|
|4092840019|4092840019|4092840019|
|3083000041|3083000041|3083000041|
|3066740023|3066740023|3066740023|
|3012820063|3012820063|3012820063|
|4054390060|4054390060|4054390060|
|3001460014|3001460014|3001460014|
|5033380056|5033380056|5033380056|
|1004630015|1004630015|1004630015|
|1018380037|1018380037|1018380037|
|3012400038|3012400038|3012400038|
|4034630001|4034630001|4034630001|
|1005390020|1005390020|1005390020|
|NULL      |NULL      |NULL      |
|1015510048|1015510048|1015510048|
|3002510011|3002510011|3002510011|
|NULL      |NULL      |NULL      |
+----------+----------+----------+
only showing top 20 rows



# 7. Create bbl_status and count each category


In [25]:
nyc311_fields_df = nyc311_fields_df.withColumn(
    "bbl_status",
    F.when(
        F.col("bbl_raw").isNull() |
        (F.trim(F.col("bbl_raw")) == ""),
        "missing"
    ).when(
        F.col("bbl").isNotNull(),
        "valid"
    ).otherwise("invalid")
)

nyc311_fields_df.groupBy("bbl_status").count().show()

[Stage 40:===================================================>  (115 + 5) / 120]

+----------+--------+
|bbl_status|   count|
+----------+--------+
|   missing| 2668612|
|     valid|20090144|
|   invalid|    2440|
+----------+--------+



# 8. Clean the Borough column

In [26]:
# Where is service demand concentrated?
# "Brooklyn " --> "BROOKLYN"

nyc311_fields_df = (
    nyc311_fields_df
    .withColumn("borough_raw", F.col("Borough"))
    .withColumn(
        "borough",
        F.when(
            F.col("Borough").isNull() |
            (F.trim(F.col("Borough")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.upper(F.trim(F.col("Borough")))
        )
    )
)

nyc311_fields_df.groupBy("borough") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(15, truncate=False)

[Stage 43:====================================================> (117 + 3) / 120]

+-------------+-------+
|borough      |count  |
+-------------+-------+
|BROOKLYN     |6833840|
|QUEENS       |5476485|
|BRONX        |4834551|
|MANHATTAN    |4580930|
|STATEN ISLAND|955909 |
|UNSPECIFIED  |41046  |
|NULL         |38433  |
|PRECINCT 112 |1      |
|PRECINCT 52  |1      |
+-------------+-------+



# 9. Create borough_status

In [27]:
# Where is service demand concentrated?

nyc311_fields_df = nyc311_fields_df.withColumn(
    "borough_status",
    F.when(
        F.col("borough").isNull(),
        "missing"
    ).when(
        F.col("borough") == "UNSPECIFIED",
        "unspecified"
    ).when(
        F.col("borough").isin(
            "BROOKLYN",
            "QUEENS",
            "BRONX",
            "MANHATTAN",
            "STATEN ISLAND"
        ),
        "valid"
    ).otherwise("unexpected")
)

nyc311_fields_df.groupBy("borough_status") \
    .count() \
    .show()

[Stage 46:=====================================================>(119 + 1) / 120]

+--------------+--------+
|borough_status|   count|
+--------------+--------+
|   unspecified|   41046|
|         valid|22681715|
|    unexpected|       2|
|       missing|   38433|
+--------------+--------+



# 10. Explore the Community Board column

In [28]:
# New York City is divided into community districts, each represented by a Community Board.
# allows to analyze service demand at a smaller geographic level than the borough
# example:
# 01 BROOKLYN — Brooklyn Community District 1
# 02 MANHATTAN — Manhattan Community District 2

nyc311_fields_df.groupBy("Community Board") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

# output: 12 BRONX = Community Board 12 in the Bronx.S

[Stage 49:=====================================================>(119 + 1) / 120]

+---------------+------+
|Community Board|count |
+---------------+------+
|12 BRONX       |847672|
|12 MANHATTAN   |719467|
|01 BROOKLYN    |564226|
|04 BRONX       |559955|
|12 QUEENS      |549996|
|05 BROOKLYN    |549676|
|01 QUEENS      |535285|
|05 QUEENS      |531821|
|07 BRONX       |513796|
|10 MANHATTAN   |505391|
|07 QUEENS      |494245|
|05 BRONX       |490017|
|03 BROOKLYN    |474464|
|14 BROOKLYN    |456151|
|02 BROOKLYN    |436985|
|07 MANHATTAN   |434362|
|17 BROOKLYN    |419867|
|09 BRONX       |418379|
|08 QUEENS      |416550|
|03 MANHATTAN   |416188|
+---------------+------+
only showing top 20 rows



# 11. Check Community Board value formats

In [29]:
nyc311_fields_df.groupBy(
    F.when(
        F.col("Community Board").isNull() |
        (F.trim(F.col("Community Board")) == ""),
        "missing"
    ).when(
        F.upper(F.trim(F.col("Community Board"))).contains("UNSPECIFIED"),
        "unspecified"
    ).when(
        F.upper(F.trim(F.col("Community Board"))).rlike(
            r"^[0-9]{2} (BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)$"
        ),
        "numbered_format"
    ).otherwise("unexpected").alias("community_board_format")
).count().show()

+----------------------+--------+
|community_board_format|   count|
+----------------------+--------+
|           unspecified|  364411|
|       numbered_format|22358348|
|               missing|   38433|
|            unexpected|       4|
+----------------------+--------+



# 12. Investigate the 4 unexpected Community Board values

In [30]:
nyc311_fields_df.filter(
    F.col("Community Board").isNotNull() &
    (F.trim(F.col("Community Board")) != "") &
    (~F.upper(F.trim(F.col("Community Board"))).contains("UNSPECIFIED")) &
    (~F.upper(F.trim(F.col("Community Board"))).rlike(
        r"^[0-9]{2} (BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)$"
    ))
).select(
    "Unique Key",
    "Borough",
    "Community Board"
).show(10, truncate=False)

[Stage 58:=====================================================>  (91 + 4) / 95]

+----------+------------+--------------------------------------------+
|Unique Key|Borough     |Community Board                             |
+----------+------------+--------------------------------------------+
|69488658  |PRECINCT 112| please call (212) NEW-YORK (212-639-9675)."|
|65647828  |PRECINCT 52 | please call (212) NEW-YORK (212-639-9675)."|
|48556313  |QUEENS      |QENB                                        |
|48187012  |QUEENS      |QENB                                        |
+----------+------------+--------------------------------------------+



# 13. Inspect other columns in these 4 records

In [31]:
nyc311_fields_df.filter(
    F.col("request_id").isin(
        "69488658",
        "65647828",
        "48556313",
        "48187012"
    )
).select(
    "Unique Key",
    "Created Date",
    "Agency",
    "Problem (formerly Complaint Type)",
    "Status",
    "Borough",
    "Community Board",
    "BBL"
).show(10, truncate=False)

[Stage 62:=======================================================>(94 + 1) / 95]

+----------+----------------------+------+---------------------------------+------+------------+--------------------------------------------+----------+
|Unique Key|Created Date          |Agency|Problem (formerly Complaint Type)|Status|Borough     |Community Board                             |BBL       |
+----------+----------------------+------+---------------------------------+------+------------+--------------------------------------------+----------+
|69488658  |06/25/2026 03:51:00 PM|DEP   |Noise                            |Closed|PRECINCT 112| please call (212) NEW-YORK (212-639-9675)."|NULL      |
|65647828  |07/24/2025 07:56:00 PM|DEP   |Noise                            |Closed|PRECINCT 52 | please call (212) NEW-YORK (212-639-9675)."|NULL      |
|48556313  |01/03/2021 06:32:19 PM|HPD   |DOOR/WINDOW                      |Closed|QUEENS      |QENB                                        |4078800300|
|48187012  |11/18/2020 12:00:00 PM|DSNY  |Derelict Vehicles                |Open  

# 14. Clean Community Board and create its quality status

In [32]:
nyc311_fields_df = (
    nyc311_fields_df

    # Preserve the original Community Board
    .withColumn(
        "community_board_raw",
        F.col("Community Board")
    )

    # Create the cleaned Community Board
    .withColumn(
        "community_board",
        F.when(
            F.upper(F.trim(F.col("Community Board"))).rlike(
                r"^[0-9]{2} (BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)$"
            ),
            F.upper(F.trim(F.col("Community Board")))
        ).otherwise(F.lit(None).cast("string"))
    )

    # Classify the Community Board quality
    .withColumn(
        "community_board_status",
        F.when(
            F.col("Community Board").isNull() |
            (F.trim(F.col("Community Board")) == ""),
            "missing"
        ).when(
            F.upper(F.trim(F.col("Community Board"))).contains("UNSPECIFIED"),
            "unspecified"
        ).when(
            F.col("community_board").isNotNull(),
            "numbered_format"
        ).otherwise("unexpected")
    )
)

nyc311_fields_df.groupBy("community_board_status").count().show()

[Stage 63:======================================================(120 + 0) / 120]

+----------------------+--------+
|community_board_status|   count|
+----------------------+--------+
|           unspecified|  364411|
|       numbered_format|22358348|
|               missing|   38433|
|            unexpected|       4|
+----------------------+--------+



# 15. Validate Community Board numbers

In [33]:
# For example, 12 BRONX is valid, but 99 BRONX has the correct text format without representing a standard community district.

# Extract the board number and borough
board_number = F.substring("community_board", 1, 2).cast("int")

board_borough = F.regexp_extract(
    "community_board",
    r"^[0-9]{2} (.+)$",
    1
)

# Define the maximum valid board number per borough
max_board = (
    F.when(board_borough == "BRONX", 12)
     .when(board_borough == "BROOKLYN", 18)
     .when(board_borough == "MANHATTAN", 12)
     .when(board_borough == "QUEENS", 14)
     .when(board_borough == "STATEN ISLAND", 3)
     .otherwise(0)
)

# Find numbered-format boards outside their valid range
nyc311_fields_df.filter(
    (F.col("community_board_status") == "numbered_format") &
    (
        (board_number < 1) |
        (board_number > max_board)
    )
).groupBy("community_board") \
 .count() \
 .orderBy(F.desc("count")) \
 .show(25, truncate=False)

[Stage 66:=====================================================>(118 + 2) / 120]

+----------------+-----+
|community_board |count|
+----------------+-----+
|64 MANHATTAN    |18928|
|83 QUEENS       |18077|
|55 BROOKLYN     |10985|
|81 QUEENS       |9577 |
|80 QUEENS       |7735 |
|26 BRONX        |6529 |
|82 QUEENS       |4287 |
|28 BRONX        |3621 |
|95 STATEN ISLAND|1836 |
|27 BRONX        |1439 |
|56 BROOKLYN     |1097 |
|84 QUEENS       |798  |
+----------------+-----+



# 16. Update community_board_status

In [34]:
special_areas = [
    "64 MANHATTAN",
    "55 BROOKLYN",
    "56 BROOKLYN",
    "26 BRONX",
    "27 BRONX",
    "28 BRONX",
    "80 QUEENS",
    "81 QUEENS",
    "82 QUEENS",
    "83 QUEENS",
    "84 QUEENS",
    "95 STATEN ISLAND"
]

nyc311_fields_df = nyc311_fields_df.withColumn(
    "community_board_status",
    F.when(
        F.col("community_board").isin(special_areas),
        "special_area"
    ).otherwise(F.col("community_board_status"))
)

nyc311_fields_df.groupBy("community_board_status").count().show()

[Stage 69:=====================================================>(118 + 2) / 120]

+----------------------+--------+
|community_board_status|   count|
+----------------------+--------+
|           unspecified|  364411|
|       numbered_format|22273439|
|          special_area|   84909|
|            unexpected|       4|
|               missing|   38433|
+----------------------+--------+



# 17. Explore the Incident Zip column

In [35]:
# Where is service demand concentrated?
# It represents the ZIP code associated with the location of a NYC 311 complaint.
# ZIP codes should remain strings, not integers

nyc311_fields_df.groupBy("Incident Zip") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

[Stage 72:====================================================> (116 + 4) / 120]

+------------+------+
|Incident Zip|count |
+------------+------+
|10466       |586792|
|11226       |353234|
|10467       |319555|
|NULL        |317725|
|10457       |310882|
|10468       |309659|
|11385       |306422|
|10452       |304790|
|10458       |286237|
|10456       |282466|
|10453       |281034|
|11207       |279690|
|11208       |257841|
|11221       |246291|
|11201       |239199|
|10031       |235789|
|11368       |225394|
|10032       |222242|
|10463       |208104|
|10462       |206828|
+------------+------+
only showing top 20 rows



# 18. Check Incident ZIP formats

In [36]:
nyc311_fields_df.groupBy(
    F.when(
        F.col("Incident Zip").isNull() |
        (F.trim(F.col("Incident Zip")) == ""),
        "missing"
    ).when(
        F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}$"),
        "five_digit"
    ).when(
        F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}-[0-9]{4}$"),
        "zip_plus_four"
    ).otherwise("unexpected").alias("zip_format")
).count().show()

[Stage 75:====================================================> (117 + 3) / 120]

+----------+--------+
|zip_format|   count|
+----------+--------+
|   missing|  317725|
|five_digit|22443446|
|unexpected|      25|
+----------+--------+



# 19. Investigate the 25 unexpected ZIP codes

In [37]:
nyc311_fields_df.filter(
    F.col("Incident Zip").isNotNull() &
    (F.trim(F.col("Incident Zip")) != "") &
    (~F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}$")) &
    (~F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}-[0-9]{4}$"))
).groupBy("Incident Zip") \
 .count() \
 .orderBy(F.desc("count")) \
 .show(30, truncate=False)

[Stage 78:=====================================================>(119 + 1) / 120]

+------------+-----+
|Incident Zip|count|
+------------+-----+
|0           |8    |
|NA          |4    |
|N/A         |3    |
|No          |1    |
|3723        |1    |
|3361        |1    |
|1155        |1    |
|na          |1    |
|1055O       |1    |
|Unsur       |1    |
|Unkno       |1    |
|1172        |1    |
|0000        |1    |
+------------+-----+



# 20. Clean Incident Zip and create its quality status

In [38]:
nyc311_fields_df = (
    nyc311_fields_df

    # Step 1: Preserve the original ZIP code
    .withColumn(
        "incident_zip_raw",
        F.col("Incident Zip")
    )

    # Step 2: Create the cleaned ZIP code
    .withColumn(
        "incident_zip",
        F.when(
            F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}$"),
            F.trim(F.col("Incident Zip"))
        ).otherwise(F.lit(None).cast("string"))
    )

    # Step 3: Create a ZIP quality status
    .withColumn(
        "incident_zip_status",
        F.when(
            F.col("Incident Zip").isNull() |
            (F.trim(F.col("Incident Zip")) == ""),
            "missing"
        ).when(
            F.col("incident_zip").isNotNull(),
            "valid_format"
        ).otherwise("invalid_format")
    )
)

# Step 4: Count records by ZIP quality status
nyc311_fields_df.groupBy("incident_zip_status").count().show()

[Stage 81:=====================================================>(119 + 1) / 120]

+-------------------+--------+
|incident_zip_status|   count|
+-------------------+--------+
|            missing|  317725|
|       valid_format|22443446|
|     invalid_format|      25|
+-------------------+--------+



# 21. Explore complaint types

In [39]:
#Which types of service complaints are most common?
#Which properties experience repeated service problems?
#Which property types experience a wide variety of city-service needs?

nyc311_fields_df.groupBy(
    "Problem (formerly Complaint Type)"
).count() \
 .orderBy(F.desc("count")) \
 .show(20, truncate=False)

[Stage 84:====================================================> (117 + 3) / 120]

+-----------------------------------+-------+
|Problem (formerly Complaint Type)  |count  |
+-----------------------------------+-------+
|Illegal Parking                    |2964367|
|Noise - Residential                |2579719|
|HEAT/HOT WATER                     |1658600|
|Noise - Street/Sidewalk            |1177444|
|Blocked Driveway                   |1086462|
|UNSANITARY CONDITION               |716077 |
|Request Large Bulky Item Collection|632148 |
|Street Condition                   |519781 |
|Water System                       |422631 |
|Noise - Vehicle                    |417036 |
|PLUMBING                           |411056 |
|Abandoned Vehicle                  |406701 |
|Noise - Commercial                 |406432 |
|PAINT/PLASTER                      |375006 |
|Noise                              |348983 |
|Street Light Condition             |307647 |
|Derelict Vehicles                  |295500 |
|Traffic Signal Condition           |286338 |
|Dirty Condition                  

# 22. Clean the complaint_type column

In [42]:
# create two new columns complaint_type_raw, complaint_type
# Identify properties with repeated service problems.
# Identify property types associated with a wide variety of city-service needs.

nyc311_fields_df = (
    nyc311_fields_df

    # Preserve the original complaint type
    .withColumn(
        "complaint_type_raw",
        F.col("Problem (formerly Complaint Type)")
    )

    # Create cleaned complaint type
    .withColumn(
        "complaint_type",
        F.when(
            F.col("complaint_type_raw").isNull() |
            (F.trim(F.col("complaint_type_raw")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("complaint_type_raw"))
        )
    )
)

nyc311_fields_df.select(
    "complaint_type_raw",
    "complaint_type"
).show(20, truncate=False)

+---------------------------+---------------------------+
|complaint_type_raw         |complaint_type             |
+---------------------------+---------------------------+
|Street Condition           |Street Condition           |
|Noise - Residential        |Noise - Residential        |
|Noise - Residential        |Noise - Residential        |
|Blocked Driveway           |Blocked Driveway           |
|Blocked Driveway           |Blocked Driveway           |
|Cannabis Retailer          |Cannabis Retailer          |
|Illegal Parking            |Illegal Parking            |
|Noise                      |Noise                      |
|Illegal Parking            |Illegal Parking            |
|Noise - Street/Sidewalk    |Noise - Street/Sidewalk    |
|Non-Emergency Police Matter|Non-Emergency Police Matter|
|Noise - Street/Sidewalk    |Noise - Street/Sidewalk    |
|Noise - Residential        |Noise - Residential        |
|Noise - Commercial         |Noise - Commercial         |
|Noise - Comme

# 23. Check missing and modified complaint types

In [43]:
nyc311_fields_df.select(
    F.sum(
        F.when(
            F.col("complaint_type").isNull(), 1
        ).otherwise(0)
    ).alias("missing_complaint_types"),

    F.sum(
        F.when(
            F.col("complaint_type_raw") != F.col("complaint_type"),
            1
        ).otherwise(0)
    ).alias("trimmed_complaint_types")
).show()

[Stage 89:=====================================================>(119 + 1) / 120]

+-----------------------+-----------------------+
|missing_complaint_types|trimmed_complaint_types|
+-----------------------+-----------------------+
|                      0|                      0|
+-----------------------+-----------------------+



# 24. Explore Problem Detail (formerly Descriptor)

In [49]:
# complaint_detail provides more specific information than complaint_type
# Identify properties with repeated service problems.
# Which properties have repeated service problems?

nyc311_fields_df.groupBy(
    "Problem Detail (formerly Descriptor)"
).count() \
 .orderBy(F.desc("count")) \
 .show(20, truncate=False)

[Stage 104:===================================================> (116 + 4) / 120]

+------------------------------------+-------+
|Problem Detail (formerly Descriptor)|count  |
+------------------------------------+-------+
|Loud Music/Party                    |3004617|
|ENTIRE BUILDING                     |1087191|
|Blocked Hydrant                     |865671 |
|No Access                           |804432 |
|Banging/Pounding                    |755109 |
|Posted Parking Sign Violation       |696878 |
|Request Large Bulky Item Collection |632148 |
|APARTMENT ONLY                      |571409 |
|N/A                                 |484590 |
|Loud Talking                        |418336 |
|With License Plate                  |406701 |
|Blocked Sidewalk                    |381471 |
|PESTS                               |376629 |
|Trash                               |357446 |
|Pothole                             |308312 |
|Derelict Vehicles                   |295500 |
|Partial Access                      |282038 |
|Car/Truck Music                     |268096 |
|Commercial O

# 25. Check missing and placeholder problem details

In [50]:
nyc311_fields_df.select(
    F.sum(
        F.when(
            F.col("Problem Detail (formerly Descriptor)").isNull(),
            1
        ).otherwise(0)
    ).alias("null_details"),

    F.sum(
        F.when(
            F.trim(F.col("Problem Detail (formerly Descriptor)")) == "",
            1
        ).otherwise(0)
    ).alias("empty_details"),

    F.sum(
        F.when(
            F.upper(F.trim(
                F.col("Problem Detail (formerly Descriptor)")
            )) == "N/A",
            1
        ).otherwise(0)
    ).alias("na_details")
).show()

[Stage 107:==================================================>  (115 + 5) / 120]

+------------+-------------+----------+
|null_details|empty_details|na_details|
+------------+-------------+----------+
|      158566|            0|    484590|
+------------+-------------+----------+



# 26. Clean the complaint_detail column

In [51]:
nyc311_fields_df = (
    nyc311_fields_df

    # Step 1: Preserve the original problem detail
    .withColumn(
        "complaint_detail_raw",
        F.col("Problem Detail (formerly Descriptor)")
    )

    # Step 2: Create the cleaned problem detail
    .withColumn(
        "complaint_detail",
        F.when(
            F.col("complaint_detail_raw").isNull() |
            (F.trim(F.col("complaint_detail_raw")) == "") |
            (F.upper(F.trim(F.col("complaint_detail_raw"))) == "N/A"),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("complaint_detail_raw"))
        )
    )
)

nyc311_fields_df.select(
    "complaint_detail_raw",
    "complaint_detail"
).show(20, truncate=False)

+---------------------------+---------------------------+
|complaint_detail_raw       |complaint_detail           |
+---------------------------+---------------------------+
|Pothole                    |Pothole                    |
|Loud Music/Party           |Loud Music/Party           |
|Loud Music/Party           |Loud Music/Party           |
|No Access                  |No Access                  |
|No Access                  |No Access                  |
|Sales to Minors Under 21   |Sales to Minors Under 21   |
|Blocked Hydrant            |Blocked Hydrant            |
|Dog                        |Dog                        |
|Parking Permit Improper Use|Parking Permit Improper Use|
|Loud Talking               |Loud Talking               |
|Other (complaint details)  |Other (complaint details)  |
|Loud Music/Party           |Loud Music/Party           |
|Loud Talking               |Loud Talking               |
|Loud Music/Party           |Loud Music/Party           |
|Loud Music/Pa

# 27. Verify the cleaned complaint_detail column

In [52]:
nyc311_fields_df.select(
    F.sum(
        F.when(
            F.col("complaint_detail").isNull(), 1
        ).otherwise(0)
    ).alias("cleaned_null_details"),

    F.sum(
        F.when(
            F.upper(F.trim(F.col("complaint_detail"))) == "N/A", 1
        ).otherwise(0)
    ).alias("remaining_na_details")
).show()

[Stage 111:===================================================> (117 + 3) / 120]

+--------------------+--------------------+
|cleaned_null_details|remaining_na_details|
+--------------------+--------------------+
|              643156|                   0|
+--------------------+--------------------+



# 28. Explore the Agency column

In [56]:
# An agency is a city department responsible for handling certain complaints.
# NYPD handles parking and noise complaints, while HPD handles housing-related complaints.
# Analyze service demand and resolution performance by agency.
# Where is resolution performance slow or inconsistent?,

nyc311_fields_df.groupBy(
    "Agency"
).count() \
 .orderBy(F.desc("count")) \
 .show(30, truncate=False)

[Stage 117:====================================================>(118 + 2) / 120]

+----------+-------+
|Agency    |count  |
+----------+-------+
|NYPD      |9958135|
|HPD       |4499910|
|DSNY      |2545306|
|DOT       |1528270|
|DEP       |1213263|
|DPR       |868505 |
|DOB       |668583 |
|DOHMH     |549764 |
|DHS       |301868 |
|TLC       |205106 |
|EDC       |178490 |
|DCWP      |127055 |
|OSE       |70955  |
|DCA       |27905  |
|DOE       |8783   |
|OOS       |4705   |
|DFTA      |2605   |
|OTI       |947    |
|DOITT     |669    |
|NYC311-PRD|371    |
|3-1-1     |1      |
+----------+-------+



# 29. Check missing values and extra spaces in Agency

In [57]:
nyc311_fields_df.select(

    # Count missing or empty agency values
    F.sum(
        F.when(
            F.col("Agency").isNull() |
            (F.trim(F.col("Agency")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_agency"),

    # Count agency values containing extra surrounding spaces
    F.sum(
        F.when(
            F.col("Agency") != F.trim(F.col("Agency")),
            1
        ).otherwise(0)
    ).alias("agency_with_extra_spaces")

).show()

[Stage 120:===================================================> (117 + 3) / 120]

+--------------+------------------------+
|missing_agency|agency_with_extra_spaces|
+--------------+------------------------+
|             0|                       0|
+--------------+------------------------+



# 30. Create the cleaned agency column

In [58]:
# create two columns - agency_raw, agency

nyc311_fields_df = (
    nyc311_fields_df

    # Step 1: Preserve original agency
    .withColumn(
        "agency_raw",
        F.col("Agency")
    )

    # Step 2: Create cleaned agency
    .withColumn(
        "agency",
        F.when(
            F.col("agency_raw").isNull() |
            (F.trim(F.col("agency_raw")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.upper(F.trim(F.col("agency_raw")))
        )
    )
)

# Display original and cleaned values
nyc311_fields_df.select(
    "agency_raw",
    "agency"
).show(20, truncate=False)

+----------+------+
|agency_raw|agency|
+----------+------+
|DOT       |DOT   |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|OOS       |OOS   |
|NYPD      |NYPD  |
|DEP       |DEP   |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|TLC       |TLC   |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|NYPD      |NYPD  |
|DHS       |DHS   |
+----------+------+
only showing top 20 rows



# 31. Verify the cleaned agency column

In [59]:
nyc311_fields_df.select(

    # Count missing cleaned agency values
    F.sum(
        F.when(
            F.col("agency").isNull(), 1
        ).otherwise(0)
    ).alias("missing_cleaned_agency"),

    # Count agency values changed during cleaning
    F.sum(
        F.when(
            F.col("agency_raw") != F.col("agency"), 1
        ).otherwise(0)
    ).alias("modified_agency"),

    # Count total requests
    F.count("*").alias("total_requests")

).show()

[Stage 124:====================================================>(118 + 2) / 120]

+----------------------+---------------+--------------+
|missing_cleaned_agency|modified_agency|total_requests|
+----------------------+---------------+--------------+
|                     0|              0|      22761196|
+----------------------+---------------+--------------+



# 32. Create an overall Silver data-quality summary

In [60]:
quality_summary_df = nyc311_fields_df.agg(

    F.count("*").alias("total_requests"),

    F.sum(F.when(F.col("bbl_status") == "missing", 1).otherwise(0))
     .alias("missing_bbl"),

    F.sum(F.when(F.col("bbl_status") == "invalid", 1).otherwise(0))
     .alias("invalid_bbl"),

    F.sum(F.when(F.col("borough_status") == "unexpected", 1).otherwise(0))
     .alias("unexpected_borough"),

    F.sum(F.when(F.col("community_board_status") == "special_area", 1).otherwise(0))
     .alias("special_area_board"),

    F.sum(F.when(F.col("community_board_status") == "unexpected", 1).otherwise(0))
     .alias("unexpected_board"),

    F.sum(F.when(F.col("incident_zip_status") == "missing", 1).otherwise(0))
     .alias("missing_zip"),

    F.sum(F.when(F.col("incident_zip_status") == "invalid_format", 1).otherwise(0))
     .alias("invalid_zip"),

    F.sum(F.when(F.col("complaint_type").isNull(), 1).otherwise(0))
     .alias("missing_complaint_type"),

    F.sum(F.when(F.col("complaint_detail").isNull(), 1).otherwise(0))
     .alias("missing_complaint_detail"),

    F.sum(F.when(F.col("agency").isNull(), 1).otherwise(0))
     .alias("missing_agency")
)

quality_summary_df.show(truncate=False, vertical=True)

[Stage 127:==================================================>  (115 + 5) / 120]

-RECORD 0----------------------------
 total_requests           | 22761196 
 missing_bbl              | 2668612  
 invalid_bbl              | 2440     
 unexpected_borough       | 2        
 special_area_board       | 84909    
 unexpected_board         | 4        
 missing_zip              | 317725   
 invalid_zip              | 25       
 missing_complaint_type   | 0        
 missing_complaint_detail | 643156   
 missing_agency           | 0        



In [ ]:
# One important finding is that not every NYC 311 request has a usable BBL.

# 33. Create a BBL join-eligibility flag

In [19]:
# Create a flag showing whether the BBL can be used for joining

nyc311_fields_df = nyc311_fields_df.withColumn(
    "bbl_join_eligible",
    F.col("bbl_status") == "valid"
)

# Count eligible and ineligible requests
nyc311_fields_df.groupBy(
    "bbl_join_eligible"
).count().show()

+-----------------+--------+
|bbl_join_eligible|   count|
+-----------------+--------+
|             true|20090144|
|            false| 2671052|
+-----------------+--------+



# 34. Create an overall row-level data-quality flag

In [18]:
# Add new column - has_invalid_field

nyc311_fields_df = nyc311_fields_df.withColumn(
    "has_invalid_field",

    (F.col("bbl_status") == "invalid") |
    (F.col("borough_status") == "unexpected") |
    (F.col("community_board_status") == "unexpected") |
    (F.col("incident_zip_status") == "invalid_format")
)

# Count requests with and without invalid fields
nyc311_fields_df.groupBy(
    "has_invalid_field"
).count().show()

[Stage 2:======================================================>(119 + 1) / 120]

+-----------------+--------+
|has_invalid_field|   count|
+-----------------+--------+
|            false|22758729|
|             true|    2467|
+-----------------+--------+



In [20]:
## restore dataframe

In [17]:
# STEP 34D: Restore all NYC 311 Silver field transformations

# Valid NYC boroughs
valid_boroughs = [
    "BRONX", "BROOKLYN", "MANHATTAN",
    "QUEENS", "STATEN ISLAND"
]

# Special-area community board codes found earlier
special_areas = [
    "64 MANHATTAN",
    "55 BROOKLYN", "56 BROOKLYN",
    "26 BRONX", "27 BRONX", "28 BRONX",
    "80 QUEENS", "81 QUEENS", "82 QUEENS",
    "83 QUEENS", "84 QUEENS",
    "95 STATEN ISLAND"
]

# Reusable expressions
bbl_text = F.trim(F.col("BBL"))
board_text = F.upper(F.trim(F.col("Community Board")))
zip_text = F.trim(F.col("Incident Zip"))
detail_text = F.trim(F.col("Problem Detail (formerly Descriptor)"))

# Rebuild the Silver DataFrame
nyc311_fields_df = (
    nyc311_raw_df

    # 1. Request ID
    .withColumn("request_id", F.trim(F.col("Unique Key")))

    # 2. BBL
    .withColumn("bbl_raw", F.col("BBL"))
    .withColumn(
        "bbl",
        F.when(
            bbl_text.rlike(r"^[1-5][0-9]{9}$"),
            bbl_text
        ).otherwise(F.lit(None).cast("string"))
    )
    .withColumn(
        "bbl_status",
        F.when(
            F.col("bbl_raw").isNull() | (bbl_text == ""),
            "missing"
        ).when(
            F.col("bbl").isNotNull(), "valid"
        ).otherwise("invalid")
    )

    # 3. Borough
    .withColumn("borough_raw", F.col("Borough"))
    .withColumn(
        "borough",
        F.when(
            F.col("Borough").isNull() |
            (F.trim(F.col("Borough")) == ""),
            F.lit(None).cast("string")
        ).otherwise(F.upper(F.trim(F.col("Borough"))))
    )
    .withColumn(
        "borough_status",
        F.when(F.col("borough").isNull(), "missing")
         .when(F.col("borough") == "UNSPECIFIED", "unspecified")
         .when(F.col("borough").isin(valid_boroughs), "valid")
         .otherwise("unexpected")
    )

    # 4. Community Board
    .withColumn("community_board_raw", F.col("Community Board"))
    .withColumn(
        "community_board",
        F.when(
            board_text.rlike(
                r"^[0-9]{2} (BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)$"
            ),
            board_text
        ).otherwise(F.lit(None).cast("string"))
    )
    .withColumn(
        "community_board_status",
        F.when(
            F.col("Community Board").isNull() |
            (F.trim(F.col("Community Board")) == ""),
            "missing"
        ).when(
            board_text.contains("UNSPECIFIED"), "unspecified"
        ).when(
            F.col("community_board").isin(special_areas), "special_area"
        ).when(
            F.col("community_board").isNotNull(), "numbered_format"
        ).otherwise("unexpected")
    )

    # 5. Incident ZIP
    .withColumn("incident_zip_raw", F.col("Incident Zip"))
    .withColumn(
        "incident_zip",
        F.when(
            zip_text.rlike(r"^[0-9]{5}$"),
            zip_text
        ).otherwise(F.lit(None).cast("string"))
    )
    .withColumn(
        "incident_zip_status",
        F.when(
            F.col("Incident Zip").isNull() | (zip_text == ""),
            "missing"
        ).when(
            F.col("incident_zip").isNotNull(), "valid_format"
        ).otherwise("invalid_format")
    )

    # 6. Complaint Type
    .withColumn(
        "complaint_type_raw",
        F.col("Problem (formerly Complaint Type)")
    )
    .withColumn(
        "complaint_type",
        F.when(
            F.col("complaint_type_raw").isNull() |
            (F.trim(F.col("complaint_type_raw")) == ""),
            F.lit(None).cast("string")
        ).otherwise(F.trim(F.col("complaint_type_raw")))
    )

    # 7. Complaint Detail
    .withColumn(
        "complaint_detail_raw",
        F.col("Problem Detail (formerly Descriptor)")
    )
    .withColumn(
        "complaint_detail",
        F.when(
            F.col("complaint_detail_raw").isNull() |
            (detail_text == "") |
            (F.upper(detail_text) == "N/A"),
            F.lit(None).cast("string")
        ).otherwise(detail_text)
    )

    # 8. Agency
    .withColumn("agency_raw", F.col("Agency"))
    .withColumn(
        "agency",
        F.when(
            F.col("agency_raw").isNull() |
            (F.trim(F.col("agency_raw")) == ""),
            F.lit(None).cast("string")
        ).otherwise(F.upper(F.trim(F.col("agency_raw"))))
    )

    # 9. BBL Join Eligibility
    .withColumn(
        "bbl_join_eligible",
        F.col("bbl_status") == "valid"
    )
)

print("Silver DataFrame restored successfully!")
print("Total columns:", len(nyc311_fields_df.columns))

Silver DataFrame restored successfully!
Total columns: 61


In [24]:
# Stop the NYC 311 Spark session
spark.stop()

print("NYC 311 Spark session stopped.")

NYC 311 Spark session stopped.
